# C4 SE path simulation (se-path-sim-v0)

Synthetic SE practice RL — **not** ASSISTments, **not** real student labels.

Run cells **in order**. Full training and final eval are gated by flags.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Dependencies


In [2]:
%pip install -q "numpy>=1.23,<3" "gymnasium>=0.29.1,<2.0" "stable-baselines3>=2.3.0,<3.0" "sb3-contrib>=2.3.0,<3.0" "torch>=2.0" "pandas>=1.5,<3" cloudpickle tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 6.8 MB/s eta 0:00:00


## 2. Upload & extract `c4-se-source-colab.zip`


In [3]:
from pathlib import Path
import zipfile, shutil
from google.colab import files

UPLOAD_DIR = Path('/content/c4_se_upload')
EXTRACT_DIR = Path('/content/c4_se_src')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

print('Upload c4-se-source-colab.zip')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded')
zip_name = next(iter(uploaded))
zip_path = UPLOAD_DIR / zip_name
zip_path.write_bytes(uploaded[zip_name])
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extracted to', EXTRACT_DIR)
print('Top:', list(EXTRACT_DIR.iterdir())[:10])


Upload c4-se-source-colab.zip


Saving c4-se-source-colab.zip to c4-se-source-colab.zip
Extracted to /content/c4_se_src
Top: [PosixPath('/content/c4_se_src/ai-services')]


## 3. Drive mount


In [4]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v0')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Artifacts root:', DRIVE_ROOT)


Artifacts root: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v0


## 4. Imports / versions / paths


In [5]:
import sys, os
from pathlib import Path

if 'EXTRACT_DIR' not in globals() or 'DRIVE_ROOT' not in globals():
    raise RuntimeError('Run cells 2 (extract) and 3 (Drive mount) first.')

hits = list(Path(EXTRACT_DIR).rglob('training_se/protocol.py'))
if not hits:
    raise FileNotFoundError('training_se/protocol.py not found under EXTRACT_DIR; re-upload ZIP')
C4_ROOT = hits[0].parents[1].resolve()
sys.path.insert(0, str(C4_ROOT))
os.chdir(C4_ROOT)
print('C4_ROOT', C4_ROOT)

from training_se.colab_bootstrap import require_symbol
import gymnasium, numpy, torch, stable_baselines3, sb3_contrib
from training_se.protocol import PROTOCOL_VERSION, GENERATOR_VERSION
print('protocol', PROTOCOL_VERSION, GENERATOR_VERSION)
print('gymnasium', gymnasium.__version__, 'sb3', stable_baselines3.__version__, 'sb3_contrib', sb3_contrib.__version__, 'torch', torch.__version__)


C4_ROOT /content/c4_se_src/ai-services/c4
protocol se-path-sim-v0 se-sim-0.1.0
gymnasium 1.3.0 sb3 2.9.0 sb3_contrib 2.9.0 torch 2.11.0+cpu


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 5. Contract / catalogue / assumptions


In [6]:
import json
from training_se.catalogue import catalogue_payload, catalogue_hash, ORDERED_TOPICS
from training_se.config import frozen_config_dict, ASSUMPTIONS
from training_se.observation import observation_schema, OBS_DIM

print('topics', ORDERED_TOPICS)
print('catalogue_sha256', catalogue_hash())
print('obs_dim', OBS_DIM)
print(json.dumps(observation_schema(), indent=2)[:800])
print(json.dumps(ASSUMPTIONS, indent=2))
print(json.dumps(frozen_config_dict(), indent=2)[:1200])


topics ('usability_heuristics', 'user_research_methods', 'prototyping', 'normalization', 'indexing', 'query_optimization', 'case_analysis_techniques', 'industry_case_studies', 'lessons_learned', 'risk_management', 'estimation', 'agile_planning')
catalogue_sha256 5001dd005443e61a27118d094ee02bd3c39fb668bbc7222dc95951e3f0624543
obs_dim 28
{
  "dim": 28,
  "dtype": "float32",
  "range": [
    0.0,
    1.0
  ],
  "order": [
    "mastery::usability_heuristics",
    "mastery::user_research_methods",
    "mastery::prototyping",
    "mastery::normalization",
    "mastery::indexing",
    "mastery::query_optimization",
    "mastery::case_analysis_techniques",
    "mastery::industry_case_studies",
    "mastery::lessons_learned",
    "mastery::risk_management",
    "mastery::estimation",
    "mastery::agile_planning",
    "available::usability_heuristics",
    "available::user_research_methods",
    "available::prototyping",
    "available::normalization",
    "available::indexing",
    "available

## 6. Environment + reproducibility checks


In [7]:
import numpy as np
from training_se.environment import make_env
from training_se.config import SeEnvConfig

cfg = SeEnvConfig()
e1 = make_env(cfg, scenario='sim_val', seed=0)
e2 = make_env(cfg, scenario='sim_val', seed=0)
o1, i1 = e1.reset(seed=123)
o2, i2 = e2.reset(seed=123)
assert np.allclose(o1, o2)
assert o1.shape == (OBS_DIM,)
assert i1['synthetic'] is True
print('repro OK', o1[:4], 'mask sum', i1['action_mask'].sum())
e1.close(); e2.close()


repro OK [0.46045434 0.32309666 0.4418301  0.41673654] mask sum 8


## 7. Smoke training + validation selection


In [8]:
from training_se.pipeline import run_training_and_validation

smoke = run_training_and_validation(
    DRIVE_ROOT,
    mode='smoke',
    timesteps=2000,
    seeds=[0],
    eval_episodes=8,
    device='cpu',
)
SMOKE_RUN_DIR = Path(smoke['run_dir'])
print('smoke run', SMOKE_RUN_DIR)
print('selected', smoke['selection']['selected'])
print('mean_val', smoke['selection']['mean_val_return_by_algorithm'])


smoke run /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v0/train_smoke_se-path-sim-v0_20261012T083335.645Z
selected maskable_ppo
mean_val {'maskable_ppo': {'mean': 8.05999980866909, 'sample_sd': 0.0}, 'weakest_stored_mastery_v1_sim': {'mean': 6.31000015884638, 'sample_sd': 0.0}, 'random_valid': {'mean': 6.660000067204237, 'sample_sd': 0.0}, 'recent_performance': {'mean': 6.660000059753656, 'sample_sd': 0.0}}


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 8. Schema-aware comparison table


In [9]:
import pandas as pd

require_symbol('SMOKE_RUN_DIR', globals().get('SMOKE_RUN_DIR'))
sel = json.loads((SMOKE_RUN_DIR / 'selection_val.json').read_text())
rows = []
for method, stats in sel['mean_val_return_by_algorithm'].items():
    rows.append({
        'method': method,
        'val_return_mean': stats['mean'],
        'val_return_sample_sd': stats['sample_sd'],
        'selected': method == sel['selected'],
        'domain': 'se_simulation',
        'synthetic': True,
        'protocol': 'se-path-sim-v0',
    })
df = pd.DataFrame(rows).sort_values('val_return_mean', ascending=False)
display(df)


,method,val_return_mean,val_return_sample_sd,selected,domain,synthetic,protocol
0,maskable_ppo,8.06,0.0,True,se_simulation,True,se-path-sim-v0
1,random_valid,6.66,0.0,False,se_simulation,True,se-path-sim-v0
2,recent_performance,6.66,0.0,False,se_simulation,True,se-path-sim-v0
3,weakest_stored_mastery_v1_sim,6.31,0.0,False,se_simulation,True,se-path-sim-v0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 9. Full training (gated)


In [10]:
# Set True only when you intend a multi-seed full run. Creates a NEW run folder.
RUN_FULL_TRAINING = True
FULL_TIMESTEPS = 50_000
FULL_SEEDS = [0, 1, 2]

FULL_RUN_DIR = None
if RUN_FULL_TRAINING:
    full = run_training_and_validation(
        DRIVE_ROOT,
        mode='full',
        timesteps=FULL_TIMESTEPS,
        seeds=FULL_SEEDS,
        eval_episodes=20,
        device='cpu',
        run_full=True,
    )
    FULL_RUN_DIR = Path(full['run_dir'])
    print('full run', FULL_RUN_DIR)
else:
    print('Skipped full training (RUN_FULL_TRAINING=False)')


full run /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v0/train_full_se-path-sim-v0_20261012T083811.995Z


In [11]:
import json
import pandas as pd

assert FULL_RUN_DIR is not None, "Run Section 9 full training first."

selection = json.loads(
    (FULL_RUN_DIR / "selection_val.json").read_text()
)

rows = [
    {
        "method": method,
        "val_return_mean": stats["mean"],
        "sample_sd_across_seeds": stats["sample_sd"],
        "selected": method == selection["selected"],
    }
    for method, stats
    in selection["mean_val_return_by_algorithm"].items()
]

display(
    pd.DataFrame(rows)
    .sort_values("val_return_mean", ascending=False)
)

print("Selected:", selection["selected"])
print("Full run folder:", FULL_RUN_DIR)
print("Evidence: synthetic SE simulator only")

,method,val_return_mean,sample_sd_across_seeds,selected
0,maskable_ppo,7.770666,0.250961,True
1,random_valid,6.716000,0.174860,False
2,recent_performance,6.492000,0.122049,False
3,weakest_stored_mastery_v1_sim,6.156000,0.296324,False


Selected: maskable_ppo
Full run folder: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v0/train_full_se-path-sim-v0_20261012T083811.995Z
Evidence: synthetic SE simulator only


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
from pathlib import Path
import json
from training_se.config import SeEnvConfig
from training_se.pipeline import run_final_evaluation

RUN_DIR = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_path_sim_v0/"
    "train_full_se-path-sim-v0_20261012T083811.995Z"
)

assert RUN_DIR.is_dir(), f"Run folder missing: {RUN_DIR}"

saved = json.loads((RUN_DIR / "train_config.json").read_text())
assert saved["protocol"] == "se-path-sim-v0"
assert saved["domain"] == "se_simulation"

# Reconstruct the actual saved training configuration.
eval_cfg = SeEnvConfig(**saved["env"])
eval_cfg.validate()

selection = json.loads(
    (RUN_DIR / "selection_val.json").read_text()
)
frozen = json.loads(
    (RUN_DIR / "SELECTION_FROZEN.json").read_text()
)

assert frozen["frozen"] is True
assert frozen["selected"] == selection["selected"]

checkpoints = list(
    RUN_DIR.glob("seed_*/maskable_ppo/maskable_ppo_final.zip")
)
assert len(checkpoints) == 3, (
    f"Expected 3 checkpoints; found {len(checkpoints)}"
)

# Avoid overwriting a previous or partially completed final evaluation.
if (RUN_DIR / "final_simulator_test").exists():
    raise RuntimeError(
        "Final evaluation folder already exists. "
        "Do not overwrite it; inspect the existing results first."
    )

print("Frozen selection:", selection["selected"])
print("Evaluating existing checkpoints; no retraining.")

plan = run_final_evaluation(
    RUN_DIR,
    run_final=True,
    eval_episodes=20,
    cfg=eval_cfg,
)

print("Final evaluation complete.")
print("Results:", RUN_DIR / "final_simulator_test")
print("Updated export:", RUN_DIR / "export_bundle")

Frozen selection: maskable_ppo
Evaluating existing checkpoints; no retraining.
Final evaluation complete.
Results: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v0/train_full_se-path-sim-v0_20261012T083811.995Z/final_simulator_test
Updated export: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v0/train_full_se-path-sim-v0_20261012T083811.995Z/export_bundle


In [13]:
import pandas as pd

summary = pd.read_csv(
    RUN_DIR / "final_simulator_test" / "test_summary.csv"
)

display(summary)
print("Evidence: synthetic SE simulation only.")
print("Final-test results are for reporting, not further tuning.")

,method,return_mean,return_sample_sd,selected_skill_gain_mean,selected_skill_gain_sample_sd,net_total_mastery_change_mean,net_total_mastery_change_sample_sd,invalid_action_rate_mean,invalid_action_rate_sample_sd
0,maskable_ppo,8.704204,0.778132,1.620871,0.078243,1.620871,0.078243,0.0,0.0
1,weakest_stored_mastery_v1_sim,7.602667,0.762368,1.511000,0.081682,1.511000,0.081682,0.0,0.0
2,random_valid,7.966365,0.770841,1.549698,0.082396,1.549698,0.082396,0.0,0.0
3,recent_performance,8.084204,0.738685,1.559204,0.077470,1.559204,0.077470,0.0,0.0


Evidence: synthetic SE simulation only.
Final-test results are for reporting, not further tuning.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile

RUN_DIR = Path(
    "/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/"
    "se_path_sim_v0/"
    "train_full_se-path-sim-v0_20261012T083811.995Z"
)
BUNDLE = RUN_DIR / "export_bundle"

manifest_path = BUNDLE / "manifest.json"
assert manifest_path.is_file(), "Export manifest missing."

manifest = json.loads(manifest_path.read_text())
hashes = manifest["sha256"]
assert hashes, "Manifest contains no file checksums."

# Verify every file recorded in the export manifest.
for relative_path, expected_hash in hashes.items():
    file_path = BUNDLE / relative_path
    assert file_path.is_file(), f"Missing: {relative_path}"
    actual_hash = hashlib.sha256(file_path.read_bytes()).hexdigest()
    assert actual_hash == expected_hash, (
        f"Checksum mismatch: {relative_path}"
    )

models = list(BUNDLE.glob("models/seed_*/maskable_ppo_final.zip"))
assert len(models) == 3, f"Expected 3 models, found {len(models)}"
assert (BUNDLE / "final_simulator_test/test_summary.csv").is_file()

# Preserve additional training metadata alongside the verified bundle.
extras = [
    RUN_DIR / "SELECTION_FROZEN.json",
    *sorted(RUN_DIR.glob("seed_*/maskable_ppo/train_meta.json")),
]
assert len(extras) == 4 and all(p.is_file() for p in extras), (
    "Expected selection marker and metadata for all three training seeds."
)

EXPORTS = RUN_DIR.parent / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ZIP_PATH = EXPORTS / f"c4_se_simulation_bundle_{stamp}.zip"

with zipfile.ZipFile(
    ZIP_PATH, "x", compression=zipfile.ZIP_DEFLATED
) as archive:
    for file_path in sorted(BUNDLE.rglob("*")):
        if file_path.is_file():
            archive.write(
                file_path,
                "export_bundle/" + file_path.relative_to(BUNDLE).as_posix(),
            )

    metadata_hashes = {}
    for file_path in extras:
        relative = file_path.relative_to(RUN_DIR).as_posix()
        archive.write(file_path, "run_metadata/" + relative)
        metadata_hashes[relative] = hashlib.sha256(
            file_path.read_bytes()
        ).hexdigest()

    archive.writestr(
        "run_metadata/checksums.json",
        json.dumps(metadata_hashes, indent=2),
    )

with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None, "ZIP integrity check failed."

print(f"Verified bundle checksums: {len(hashes)}/{len(hashes)}")
print("Model checkpoints:", len(models))
print("Package saved:", ZIP_PATH)
print("Size MB:", round(ZIP_PATH.stat().st_size / 1024**2, 2))

Verified bundle checksums: 26/26
Model checkpoints: 3
Package saved: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_sim_v0/exports/c4_se_simulation_bundle_20261012T084659312700Z.zip
Size MB: 0.5


In [15]:
from google.colab import files
files.download(str(ZIP_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Final simulator evaluation (gated; after frozen selection)


In [ ]:
from training_se.pipeline import run_final_evaluation

RUN_FINAL_EVALUATION = False
# Point at a run that already has SELECTION_FROZEN.json (full or smoke).
FINAL_TARGET_RUN = FULL_RUN_DIR or globals().get('SMOKE_RUN_DIR')

if RUN_FINAL_EVALUATION:
    require_symbol('FINAL_TARGET_RUN', FINAL_TARGET_RUN)
    plan = run_final_evaluation(FINAL_TARGET_RUN, run_final=True, eval_episodes=20)
    print(plan)
else:
    print('Skipped final evaluation (RUN_FINAL_EVALUATION=False)')


## 11. Artifact export reminder


In [ ]:
require_symbol('SMOKE_RUN_DIR', globals().get('SMOKE_RUN_DIR'))
exp = SMOKE_RUN_DIR / 'export_bundle'
print('Export bundle:', exp)
print('Files:', sorted(p.name for p in exp.iterdir()) if exp.is_dir() else 'missing')
print('Label: simulation_only / domain=se_simulation / no readiness claims')
print('Manifest:', (exp / 'manifest.json').is_file())
